In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DoubleType, BooleanType, DateType, TimestampType
)
spark = SparkSession.builder.appName("ecommerceData").getOrCreate()


In [0]:
import os
from dotenv import load_dotenv

load_dotenv()

storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")
storage_account_access_key = os.getenv("STORAGE_ACCOUNT_ACCESS_KEY")


In [0]:
import os
from dotenv import load_dotenv

load_dotenv("01_bronze/.env")

storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
container_name = os.getenv("CONTAINER_NAME")
storage_account_access_key = os.getenv("STORAGE_ACCOUNT_ACCESS_KEY")

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/"
all_files = ["Olist-Dataset/olist_customers_dataset.csv",
             "Olist-Dataset/olist_geolocation_dataset.csv",
             "Olist-Dataset/olist_order_items_dataset.csv",
             "Olist-Dataset/olist_order_payments_dataset.csv",
             "Olist-Dataset/olist_order_reviews_dataset.csv",
             "Olist-Dataset/olist_orders_dataset.csv",
             "Olist-Dataset/olist_products_dataset.csv",
             "Olist-Dataset/olist_sellers_dataset.csv",
             "Olist-Dataset/product_category_name_translation.csv"]
catalog_name = "de_mini_project"
schema_name = "01_bronze_schema"

for i in all_files:
    full_path = base_path + f"{i}"
    df = (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_access_key)
        .load(full_path)
    )
    df = df.write.format("delta").saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")
    table_name = i.split("/")[-1].replace(".csv", "")
    